### Clustering prediction

In [8]:
import os
print(os.getcwd())


d:\Downloads


In [37]:
import joblib

#  Correct way: Load using the actual full paths
kmeans = joblib.load(r"D:\Downloads\kmeans_model.pkl")
scaler = joblib.load(r"D:\Downloads\scaler.pkl")

print(" Model and scaler loaded successfully!")


 Model and scaler loaded successfully!


In [ ]:
import pandas as pd
import numpy as np
import joblib


# Load Pre-trained Model and Scaler

print(" Loading model and scaler...")

kmeans = joblib.load(r"D:\Downloads\kmeans_model.pkl")
scaler = joblib.load(r"D:\Downloads\scaler.pkl")

print(" Model and scaler loaded successfully!\n")


#  Load Customer Data

data_path = r"D:\Downloads\customer_data_with_products.csv"

#  encodings 
encodings_to_try = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'windows-1252']

df = None
for encoding in encodings_to_try:
    try:
        print(f"Trying encoding: {encoding}...")
        df = pd.read_csv(data_path, encoding=encoding)
        print(f" Data loaded successfully with encoding: {encoding}")
        break
    except UnicodeDecodeError:
        continue
    except Exception as e:
        print(f"Error with {encoding}: {e}")
        continue

if df is None:
    raise ValueError(" Could not read the CSV file with any common encoding")

print(f" Data loaded successfully. Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

# Prepare Features

features = ['Recency', 'Avg_Order_Value', 'Customer_Lifetime_Days', 'Purchase_Rate', 'Total_Items_Sold']

# Validate features exist
missing_features = [f for f in features if f not in df.columns]
if missing_features:
    raise ValueError(f" Missing feature columns: {missing_features}")

print(f"\n All required features found: {features}")

# 4️ Scale Data and Predict Clusters

X = df[features]
X_scaled = scaler.transform(X)

df["Predicted_Cluster"] = kmeans.predict(X_scaled)


# 5️ Map Clusters to Descriptive Names

cluster_mapping = {
    0: "Dormant/Churned",
    1: "Loyal/Engaged",
    2: "New/Recent but Inactive",
    3: "High-Engagement/Recent High-Value"
}

df["Cluster_Description"] = df["Predicted_Cluster"].map(cluster_mapping)

print("\n Cluster Prediction Summary:")
print(df["Cluster_Description"].value_counts())
print("\n" + "="*60)

# Analyze Top Products per Cluster

print("\n Analyzing top products by cluster...")

product_cols = [col for col in df.columns if col.startswith('Product_')]

if not product_cols:
    print(" No product columns found (columns starting with 'Product_').")
    top_products_by_cluster = None
else:
    print(f" Found {len(product_cols)} product columns\n")
    
    # Melt product columns for analysis
    df_melted = df.melt(
        id_vars=['Cluster_Description'],
        value_vars=product_cols,
        var_name='Product_Column',
        value_name='Product'
    ).dropna(subset=['Product'])
    
    # Count product occurrences per cluster
    product_counts = (
        df_melted.groupby(['Cluster_Description', 'Product'])
        .size()
        .reset_index(name='Count')
        .sort_values(['Cluster_Description', 'Count'], ascending=[True, False])
    )
    
    # Extract top products per cluster
    top_products_by_cluster = {}
    
    print("🏆 Top 10 Products per Cluster:")
    print("="*60)
    
    for cluster_name in sorted(df['Cluster_Description'].unique()):
        cluster_products = (
            product_counts[product_counts['Cluster_Description'] == cluster_name]
            .sort_values('Count', ascending=False)
            .head(10)
        )
        top_products_by_cluster[cluster_name] = cluster_products
        
        print(f"\n {cluster_name}:")
        print(cluster_products.to_string(index=False))


#  Save Results

print("\n" + "="*60)
print(" Saving results...")

# Save CSV with predictions
csv_output = r"D:\Downloads\cluster_results.csv"
df.to_csv(csv_output, index=False)
print(f" CSV saved: {csv_output}")

# Save Excel with multiple sheets
excel_output = r"D:\Downloads\cluster_results_with_products.xlsx"

with pd.ExcelWriter(excel_output, engine='openpyxl') as writer:
    # Main predictions sheet
    df.to_excel(writer, sheet_name='Cluster_Predictions', index=False)
    
    # Top products sheets (one per cluster)
    if top_products_by_cluster:
        for cluster_name, table in top_products_by_cluster.items():
            # Clean sheet name (Excel has 31 char limit)
            sheet_name = cluster_name.replace("/", "_").replace(" ", "_")[:31]
            table.to_excel(writer, sheet_name=f"Top_{sheet_name}", index=False)

print(f" Excel saved: {excel_output}")


#  Summary Statistics

print("\n" + "="*60)
print(" SUMMARY STATISTICS BY CLUSTER")
print("="*60)

summary = df.groupby('Cluster_Description')[features].agg(['mean', 'median', 'count'])
print(summary)

print("\n✨ Analysis complete!")

## Retention prediction

In [ ]:
import pandas as pd
import numpy as np
import joblib


# CONFIGURATION

MODEL_PATH = r"D:\Downloads\customer_retention_model.pkl"
DATA_PATH = r"D:\Downloads\dummy_customer_data.csv"
OUTPUT_PATH = r"C:\Users\DELL\Desktop\agroX\agroX\customer_retention_predictions.xlsx"

FEATURE_COLUMNS = [
    'Recency_y', 'Frequency', 'Monetary', 'Customer_Lifetime_Days', 'Purchase_Rate',
    'Total_Items_Sold', 'Unique_Products_Count_y', 'Avg_Items_Per_Order_y',
    'Avg_Revenue_Per_Order_y', 'Avg_Net_Sales_Per_Order_y'
]


# FUNCTION DEFINITIONS


def load_csv_with_encoding(file_path):
    """Load CSV file trying multiple encodings."""
    encodings_to_try = [
        ('cp1252', 'Windows-1252 (Western European)'),
        ('utf-8', 'UTF-8'),
        ('latin1', 'Latin-1'),
        ('iso-8859-1', 'ISO-8859-1'),
        ('cp850', 'CP850 (DOS)'),
        ('utf-16', 'UTF-16')
    ]
    
    print(f" Attempting to load: {file_path}")
    
    for encoding, description in encodings_to_try:
        try:
            print(f"   Trying {description}...", end='')
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"  Success!")
            print(f"CSV loaded successfully with {encoding} encoding")
            return df
        except UnicodeDecodeError:
            print(f"  Failed")
            continue
        except Exception as e:
            print(f"  Error: {e}")
            continue
    
    raise ValueError(" Could not load CSV with any common encoding. Please check the file.")


def load_model(model_path):
    """Load the trained model."""
    try:
        model = joblib.load(model_path)
        print(" Model loaded successfully")
        return model
    except FileNotFoundError:
        raise FileNotFoundError(f" Model file not found at: {model_path}")
    except Exception as e:
        raise Exception(f" Error loading model: {e}")


def validate_features(df, required_features):
    """Validate that all required features exist in the dataframe."""
    missing_features = [col for col in required_features if col not in df.columns]
    
    if missing_features:
        print(f"\nMissing features: {missing_features}")
        print(f"\n Available columns in dataset:")
        for i, col in enumerate(df.columns.tolist(), 1):
            print(f"   {i}. {col}")
        raise ValueError(f"Missing required features: {missing_features}")
    
    print(f" All {len(required_features)} required features found")
    return True


def preprocess_features(df, features):
    """Prepare features for prediction."""
    X = df[features].copy()
    
    # Handle infinite values
    inf_count = np.isinf(X).sum().sum()
    if inf_count > 0:
        print(f" Found {inf_count} infinite values - replacing with NaN")
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Count and display missing values
    missing_counts = X.isnull().sum()
    total_missing = missing_counts.sum()
    
    if total_missing > 0:
        print(f"\n Found {total_missing} missing values:")
        for col, count in missing_counts[missing_counts > 0].items():
            print(f"   • {col}: {count} ({count/len(X)*100:.1f}%)")
        print("   Filling with column means...")
        X.fillna(X.mean(), inplace=True)
    else:
        print(" No missing values found")
    
    print(f" Features preprocessed: {X.shape[0]} rows × {X.shape[1]} features")
    return X


def make_predictions(model, X):
    """Generate predictions using the model."""
    try:
        predictions = model.predict(X)
        print(f" Predictions generated for {len(predictions)} records")
        return predictions
    except Exception as e:
        raise Exception(f" Error making predictions: {e}")


def analyze_top_products(df):
    """Analyze top products by retention status."""
    product_cols = [col for col in df.columns if col.startswith('Product_')]
    
    if not product_cols:
        print(" No product columns found (columns starting with 'Product_')")
        return None
    
    print(f"\n Found {len(product_cols)} product columns")
    print(f"   Analyzing product preferences...")
    
    # Melt product columns into long format
    df_melted = df.melt(
        id_vars=['Retention_Status'],
        value_vars=product_cols,
        var_name='Product_Column',
        value_name='Product'
    ).dropna(subset=['Product'])
    
    # Remove empty strings and whitespace
    df_melted['Product'] = df_melted['Product'].astype(str).str.strip()
    df_melted = df_melted[df_melted['Product'] != '']
    
    if len(df_melted) == 0:
        print(" No product data found after cleaning")
        return None
    
    # Count products by retention status
    product_counts = (
        df_melted.groupby(['Retention_Status', 'Product'])
        .size()
        .reset_index(name='Count')
        .sort_values(['Retention_Status', 'Count'], ascending=[True, False])
    )
    
    # Get top products for each retention status
    top_products_by_class = {}
    for status in sorted(df['Retention_Status'].unique()):
        top_products = (
            product_counts[product_counts['Retention_Status'] == status]
            .sort_values('Count', ascending=False)
            .head(20)  # Get top 20 for each class
        )
        top_products_by_class[status] = top_products
        
        print(f"\n Top 10 Products for '{status}' Customers:")
        if len(top_products) > 0:
            display_df = top_products.head(10).copy()
            display_df.index = range(1, len(display_df) + 1)
            print(display_df.to_string())
        else:
            print("   No products found")
    
    return top_products_by_class


def save_to_excel(df, top_products_by_class, output_path):
    """Save predictions and analysis to Excel file."""
    try:
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            # Save main predictions
            df.to_excel(writer, sheet_name='Predictions', index=False)
            print(f"\n Saved 'Predictions' sheet: {len(df)} rows")
            
            # Create summary sheet
            summary_data = {
                'Metric': [
                    'Total Customers',
                    'Returning Customers',
                    'Not Returning Customers',
                    'Retention Rate (%)',
                    'Churn Rate (%)'
                ],
                'Value': [
                    len(df),
                    (df['Retention_Status'] == 'Returning').sum(),
                    (df['Retention_Status'] == 'Not Returning').sum(),
                    f"{(df['Retention_Status'] == 'Returning').sum() / len(df) * 100:.2f}",
                    f"{(df['Retention_Status'] == 'Not Returning').sum() / len(df) * 100:.2f}"
                ]
            }
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_excel(writer, sheet_name='Summary', index=False)
            print(f" Saved 'Summary' sheet")
            
            # Save top products for each retention class
            if top_products_by_class:
                for status, table in top_products_by_class.items():
                    sheet_name = f"Top_{status.replace(' ', '_')}"[:31]  # Excel sheet name limit
                    table.to_excel(writer, sheet_name=sheet_name, index=False)
                    print(f"💾 Saved '{sheet_name}' sheet: {len(table)} products")
        
        print(f"\n All results saved successfully to:")
        print(f"    {output_path}")
    except PermissionError:
        raise Exception(f"Permission denied. Please close the Excel file if it's open:\n   {output_path}")
    except Exception as e:
        raise Exception(f" Error saving to Excel: {e}")


def display_statistics(df):
    """Display prediction statistics."""
    print(f"\n{'='*60}")
    print(" PREDICTION STATISTICS")
    print(f"{'='*60}")
    
    total = len(df)
    returning = (df['Retention_Status'] == 'Returning').sum()
    not_returning = (df['Retention_Status'] == 'Not Returning').sum()
    
    print(f"\n Overall Statistics:")
    print(f"   Total Customers:        {total:,}")
    print(f"   Returning Customers:    {returning:,} ({returning/total*100:.2f}%)")
    print(f"   Not Returning:          {not_returning:,} ({not_returning/total*100:.2f}%)")
    print(f"   Retention Rate:         {returning/total*100:.2f}%")
    print(f"   Churn Rate:             {not_returning/total*100:.2f}%")
    
    # Display sample predictions
    print(f"\n Sample Predictions (First 10 Customers):")
    sample_cols = ['Retention_Status'] + FEATURE_COLUMNS[:5]
    available_cols = [col for col in sample_cols if col in df.columns]
    sample_df = df[available_cols].head(10).copy()
    sample_df.index = range(1, len(sample_df) + 1)
    print(sample_df.to_string())



# MAIN EXECUTION


def main():
    """Main execution function."""
    print("\n" + "="*60)
    print(" CUSTOMER RETENTION PREDICTION PIPELINE")
    print("="*60)
    
    try:
        # Step 1: Load model
        print("\n[Step 1/6] Loading trained model...")
        model = load_model(MODEL_PATH)
        
        # Step 2: Load data
        print("\n[Step 2/6] Loading customer data...")
        df = load_csv_with_encoding(DATA_PATH)
        print(f" Data shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
        
        # Step 3: Validate features
        print("\n[Step 3/6] Validating features...")
        validate_features(df, FEATURE_COLUMNS)
        
        # Step 4: Preprocess features
        print("\n[Step 4/6] Preprocessing features...")
        X = preprocess_features(df, FEATURE_COLUMNS)
        
        # Step 5: Make predictions
        print("\n[Step 5/6] Making predictions...")
        predictions = make_predictions(model, X)
        
        # Add predictions to dataframe
        df['Predicted_Retention'] = predictions
        df['Retention_Status'] = df['Predicted_Retention'].map({
            1: 'Returning',
            0: 'Not Returning'
        })
        
        # Display statistics
        display_statistics(df)
        
        # Step 6: Analyze top products
        print("\n[Step 6/6] Analyzing top products by retention class...")
        top_products_by_class = analyze_top_products(df)
        
        # Save results
        print(f"\n{'='*60}")
        print(" SAVING RESULTS")
        print(f"{'='*60}")
        save_to_excel(df, top_products_by_class, OUTPUT_PATH)
        
        print(f"\n{'='*60}")
        print(" PIPELINE COMPLETED SUCCESSFULLY!")
        print(f"{'='*60}\n")
        
    except Exception as e:
        print(f"\n{'='*60}")
        print(f" PIPELINE FAILED")
        print(f"{'='*60}")
        print(f"Error: {e}\n")
        import traceback
        traceback.print_exc()





if __name__ == "__main__":
    main()